In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
import sys
import asyncio

# Fix for Windows issues in Jupyter notebooks
if sys.platform == "win32":
    # 1. Use ProactorEventLoop for subprocess support
    if not isinstance(asyncio.get_event_loop_policy(), asyncio.WindowsProactorEventLoopPolicy):
        asyncio.set_event_loop_policy(asyncio.WindowsProactorEventLoopPolicy())
    
    # 2. Redirect stderr to avoid fileno() error when launching MCP servers
    if "ipykernel" in sys.modules:
        sys.stderr = sys.__stderr__


## Local MCP server

In [3]:
from langchain_mcp_adapters.client import MultiServerMCPClient

client = MultiServerMCPClient(
    {
        "local_server": {
                "transport": "stdio",
                "command": "python",
                "args": ["resources/2.1_mcp_server.py"],
            }
    }
)

In [4]:
# get tools
tools = await client.get_tools()

# get resources
resources = await client.get_resources("local_server")

# get prompts
prompt = await client.get_prompt("local_server", "prompt")
prompt = prompt[0].content

In [5]:
from langchain.agents import create_agent

agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
    system_prompt=prompt
)

In [6]:
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = await agent.ainvoke(
    {"messages": [HumanMessage(content="Tell me about the langchain-mcp-adapters library")]},
    config=config
)

In [8]:
from pprint import pprint

pprint(response['messages'][-1].content)

('Here’s a concise overview of the langchain-mcp-adapters library and how it '
 'fits into LangChain, LangGraph, and LangSmith.\n'
 '\n'
 'What it is\n'
 '- A adapters library that exposes Anthropic’s Model Context Protocol (MCP) '
 'tools to LangChain/LangGraph workflows.\n'
 '- It converts MCP tools into LangChain-ready tools and lets you connect to '
 'one or multiple MCP servers from a LangChain agent flow.\n'
 '\n'
 'What MCP is (brief)\n'
 '- MCP (Model Context Protocol) is an open protocol for structured '
 'interactions between language models and tool servers. It lets tools from '
 'MCP servers be discovered, invoked, and possibly require user input (forms, '
 'URLs, etc.) during a run.\n'
 '\n'
 'Key features of the adapters\n'
 '- Convert MCP tools into LangChain tools so they can be used in LangChain '
 'agents and LangGraph workflows.\n'
 '- A client that can connect to multiple MCP servers and load their tools.\n'
 '- Optional tool name prefixing to avoid name clashes whe

## Online MCP

In [9]:
client = MultiServerMCPClient(
    {
        "time": {
            "transport": "stdio",
            "command": "uv",
            "args": [
                "run",
                "python",
                "-m",
                "mcp_server_time",
                "--local-timezone=America/New_York"
            ]
        }
    }
)

tools = await client.get_tools()

In [10]:
agent = create_agent(
    model="gpt-5-nano",
    tools=tools,
)

In [12]:
question = HumanMessage(content="What time is it?")

response = await agent.ainvoke(
    {"messages": [question]}
)

pprint(response['messages'][-1].content)

('In New York (Eastern Time): 8:41 AM on Thursday, October 29, 2026 (Eastern '
 'Daylight Time). Want this in another time zone?')
